In [ ]:
def random_rotations(B, generator, reflections=True):
    A = torch.randn(B, 3, 3, generator=generator)
    Q, R = torch.linalg.qr(A)
    Q = Q * torch.sign(torch.diagonal(R, dim1=1, dim2=2)).unsqueeze(1)   # Uniforme sur les rotations
    if not reflections:
        Q[torch.linalg.det(Q) < 0, :, 0] *= -1                          # On garde det = 1
    return Q

def invariance_error(faenet, loader, mu_target_train, std_target_train, n_rotations=10, seed=0):
    faenet.eval()
    generator = torch.Generator().manual_seed(seed)
    stds = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            preds = []
            for _ in range(n_rotations):
                Q = random_rotations(batch.num_graphs, generator).to(device)
                rotated = batch.clone()
                rotated.pos = torch.bmm(batch.pos.unsqueeze(1), Q[batch.batch]).squeeze(1)
                _, y = faenet(rotated, sfa_generator=generator)
                preds.append(mu_target_train + std_target_train * y)
            preds = torch.cat(preds, dim=1)          # (n_molecules_in_batch, n_rotations)
            stds.append(preds.std(dim=1).cpu())      # Ecarts types pour chaque molécule
    stds = torch.cat(stds)
    return stds.mean().item(), stds.max().item(), stds
